# High-Performance Embeddings

### Why is embedding performance crucial?
- **Search Quality**: The accuracy of your RAG directly depends on the quality of the embeddings.
- **Operational Cost**: Local models can drastically reduce large-scale API costs.
- **Speed (Latency)**: The time to generate embeddings impacts indexing speed and user response time.
- **Privacy**: Local models ensure sensitive data remains within your infrastructure.

## Settings

In [1]:
!pip install langchain langchain-google-genai sentence-transformers scikit-learn langchain-community

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached protobuf-6.32.1-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 109.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 103.2 MB/s eta 0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached protobuf-6.32.1-cp39-abi3-macosx_10_9_universal2.whl (426 kB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.2
    Uninstalling sympy-1.13.2:
      Successfully uninstalled sympy-1.13.2
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ortools 9.12.4544 requires protobuf<5.30,>=5.29.3, but you have 

## Setup - Load API Key and Initialize Client

In [2]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print("API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

API key loaded successfully.


## Model Comparison: OpenAI (API) vs. Hugging Face (Local)

Let's compare a cutting-edge model via API (OpenAI) with popular open-source models that run locally.

- **Open AI**: High-quality model, accessed via API.
- **`all-MiniLM-L6-v2`**: A very popular, lightweight, and fast local model. Great for general tasks where speed is important.
- **`BAAI/bge-large-en-v1.5`**: One of the best open-source models on the MTEB Leaderboard. Heavier, but with superior semantic quality.

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import time

In [5]:
# Sample texts for our test
test_texts = [
    "What is our company's vacation policy?",
    "I need a travel expense report.",
    "How do I set up access to the virtual private network (VPN)?",
    "Where can I find the organization's code of conduct?",
    "I want to understand the performance evaluation process."
]

#### OpenAI Embeddings

In [6]:
openai_embeddings = OpenAIEmbeddings(openai_api_key=api_key)

start_time = time.time()
embeddings_gemini = openai_embeddings.embed_documents(test_texts)
end_time = time.time()

print(f"Processing time: {end_time - start_time} seconds")
print(f"  - Vector dimensions: {len(embeddings_gemini[0])}")

Processing time: 0.9468789100646973 seconds
  - Vector dimensions: 1536


#### Hugging Face Embeddings - All-MiniLM-L6-v2

In [9]:

minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
start_time = time.time()
embeddings_minilm = minilm_embeddings.embed_documents(test_texts)
end_time = time.time()

print(f"Processing time: {end_time - start_time} seconds")
print(f"  - Vector dimensions: {len(embeddings_minilm[0])}")

/Users/julio.cesar/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing time: 1.164687156677246 seconds
  - Vector dimensions: 384


[HuggingFaceEmbeddings](https://python.langchain.com/api_reference/huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings.html)

[all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
)

#### Hugging Face Embeddings - BGE-Large

In [10]:
bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

start_time = time.time()
embeddings_bge = bge_embeddings.embed_documents(test_texts)
end_time = time.time()

print(f"Processing time: {end_time - start_time} seconds")
print(f"  - Vector dimensions: {len(embeddings_gemini[0])}")

Processing time: 0.2209608554840088 seconds
  - Vector dimensions: 1536


[bge-large-en-v1.5](https://huggingface.co/BAAI/bge-large-en-v1.5)

### Semantic Quality Analysis

Now, let's see which model better understands a semantically similar question, but with different words.

In [12]:
question = "I want to take a few days off work."

emb_question_openai = openai_embeddings.embed_query(question)
emb_question_minilm = minilm_embeddings.embed_query(question)
emb_question_bge = bge_embeddings.embed_query(question)

In [16]:
models = {
    "OpenAI": (emb_question_openai, embeddings_gemini),
    "MiniLM": (emb_question_minilm, embeddings_minilm),
    "BGE-large": (emb_question_bge, embeddings_bge)
}

print(question)

I want to take a few days off work.


In [17]:
for name, (emb_q, emb_docs) in models.items():
  similarities = cosine_similarity([emb_q], emb_docs)[0]
  doc_e_similarity = sorted(
      zip(test_texts, similarities), key=lambda x: x[1], reverse=True
  )
  print(f"--- Ranking para o modelo {name} ---")
  for i, (doc, sim) in enumerate(doc_e_similarity[:3], 1):
    print(f"  {i}. (Score: {sim:.3f}) {doc}")
  print()

--- Ranking para o modelo OpenAI ---
  1. (Score: 0.820) What is our company's vacation policy?
  2. (Score: 0.808) I need a travel expense report.
  3. (Score: 0.746) I want to understand the performance evaluation process.

--- Ranking para o modelo MiniLM ---
  1. (Score: 0.319) What is our company's vacation policy?
  2. (Score: 0.306) I need a travel expense report.
  3. (Score: 0.103) Where can I find the organization's code of conduct?

--- Ranking para o modelo BGE-large ---
  1. (Score: 0.646) What is our company's vacation policy?
  2. (Score: 0.549) I need a travel expense report.
  3. (Score: 0.464) I want to understand the performance evaluation process.



## Caching Embeddings: Cost and Speed

Generating embeddings, especially via API, has time and money costs. The cache stores already calculated embeddings to avoid rework. We will use LangChain's `CacheBackedEmbeddings`.

In [18]:
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings

store = LocalFileStore("./cache/")

embedder_principal = OpenAIEmbeddings(openai_api_key=api_key)

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embedder_principal,
    store,
    namespace='openai_cache'
)

/Users/julio.cesar/Library/Python/3.9/lib/python/site-packages/langchain/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [20]:
import time

texts_for_caching = ["This is a test document number 1.",
                     "This is a test document number 2.",
                     "This is a test document number 3."]

start_time = time.time()
embeddings_result_1 = cache_embeddings.embed_documents(test_texts)
end_time = time.time()

print(f"  - Execution time: {end_time - start_time:.4f} seconds.")

  - Execution time: 0.4042 seconds.


## Batch Processing for Large-Scale Indexing

When indexing thousands of documents, processing them in batches is essential. Local models, in particular, benefit immensely from this.

In [21]:
large_document = [f"This is document test number: {i}." for i in range(1000)]

bge_embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

batch_sizes = [1, 32, 64, 128]

In [22]:
for batch_size in batch_sizes:
    start_time = time.time()
    num_batches = len(large_document) // batch_size
    estimated_time = num_batches * (0.1 * batch_size) + (len(large_document) % batch_size) * 0.1
    real_time = bge_embedder.client.encode(large_document, batch_size=batch_size)
    end_time = time.time()

    print(f"  - Batch Size: {batch_size:<4} -> Tempo: {end_time - start_time:.2f}s")

  - Batch Size: 1    -> Tempo: 96.14s
  - Batch Size: 32   -> Tempo: 5.69s
  - Batch Size: 64   -> Tempo: 4.65s
  - Batch Size: 128  -> Tempo: 3.99s
